# 3.0 — Evaluación y calibración del clasificador de lluvia

Este notebook **no entrena nada**: lee los artefactos que generan `make study` y `make train`
(`reports/metrics/rain_classifier/...`) y cuenta cómo se tomó cada decisión. Para regenerarlo:
`make study train notebook`.

**Pregunta**: ¿lloverá (≥ 0.5 mm) en la celda ERA5 de un punto del Valle de Aburrá en una hora dada, a partir del
pronóstico de Open-Meteo para ese punto? **Reglas del juego**: toda decisión se toma con validación cruzada temporal
dentro de 2022–2024; 2025 se usó para reportar la versión 1.0.0 y 2026 es un holdout que no tocó ninguna decisión.

In [1]:
import json
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
M = RAIZ / "reports" / "metrics" / "rain_classifier"
study = json.loads((M / "study" / "study.json").read_text(encoding="utf-8"))
version = sorted(p.name for p in M.iterdir() if p.name[0].isdigit())[-1]
metrics = json.loads((M / version / "metrics.json").read_text(encoding="utf-8"))
pd.set_option("display.precision", 4)
print("Modelo", version, "· desarrollo", study["development"]["start"][:10], "→", study["development"]["end"][:10],
      f"· {study['development']['rows']:,} filas · tasa de lluvia {study['development']['rain_rate']:.1%}")
pd.DataFrame(study["folds"])

Modelo 1.1.0 · desarrollo 2022-01-01 → 2024-12-31 · 322,894 filas · tasa de lluvia 18.0%


,name,train_rows,valid_rows,valid_start,valid_end
0,2023-T3,150982,28704,2023-07-01T00:00:00+00:00,2023-09-30T23:00:00+00:00
1,2023-T4,179686,28704,2023-10-01T00:00:00+00:00,2023-12-31T23:00:00+00:00
2,2024-T1,208390,28392,2024-01-01T00:00:00+00:00,2024-03-31T23:00:00+00:00
3,2024-T2,236782,28392,2024-04-01T00:00:00+00:00,2024-06-30T23:00:00+00:00
4,2024-T3,265174,28704,2024-07-01T00:00:00+00:00,2024-09-30T23:00:00+00:00
5,2024-T4,293878,28704,2024-10-01T00:00:00+00:00,2024-12-31T23:00:00+00:00


## 1. Validación cruzada *rolling origin*

Seis folds trimestrales con ventana de entrenamiento creciente y 24 h de hueco: la autocorrelación horaria haría
optimista un K-fold aleatorio, y el modelo siempre predice el futuro respecto de lo que vio.

## 2. ¿Qué features agregar? (con la configuración de referencia 1.0.0)

La ventana del pronóstico (t−2…t+2) absorbe errores de *timing* del modelo numérico; la ubicación captura gradientes
dentro del valle. La ubicación se acepta solo si también mejora en validación **espacial** (entrenar sin una celda ERA5
y evaluar en ella), porque el producto predice en puntos arbitrarios.

In [2]:
sets = pd.DataFrame({k: {"log loss CV": v["cv"]["mean"]["log_loss"], "Δ vs base": v["vs_base"]["mean"],
                         "EE": v["vs_base"]["se"], "log loss espacial": study["spatial_cv"][k]["mean_log_loss"]}
                     for k, v in study["feature_candidates"]["sets"].items()}).T
marginal = pd.DataFrame(study["feature_candidates"]["marginal"]).T[["mean", "se", "better_by_1se"]]
display(sets)
marginal

,log loss CV,Δ vs base,EE,log loss espacial
base,0.3088,0.0000,0.0000,0.3319
base+window,0.3026,-0.0062,0.0015,0.3209
base+location,0.3006,-0.0082,0.0013,0.3227
base+window+location,0.2964,-0.0123,0.0014,0.3110


,mean,se,better_by_1se
window|base,-0.0062,0.0015,True
window|location,-0.0041,0.0013,True
location|base,-0.0082,0.0013,True
location|window,-0.0062,0.0007,True


## 3. Búsqueda de hiperparámetros

Búsqueda aleatoria (semilla fija) de 60 configuraciones de HistGradientBoosting más dos familias de referencia
(logística con *splines*, un GAM aditivo, y bosque aleatorio, la familia del modelo heredado). Se elige con la regla de
**un error estándar**: entre las configuraciones a ≤ 1 EE de la mejor, la más simple.

![Sensibilidad](../reports/figures/study_hgb_sensitivity.png)

In [3]:
t = pd.json_normalize(study["search"]["trials"])
t["ll"] = t["mean.log_loss"]
print("Elegida:", study["search"]["chosen"]["family"], study["search"]["chosen"]["params"])
print("Δ vs configuración 1.0.0 (misma CV):", study["search"]["chosen_vs_v1"])
t.groupby("family")["ll"].describe()[["count", "min", "50%", "max"]]

Elegida: hgb {'learning_rate': 0.097271, 'max_leaf_nodes': 29, 'min_samples_leaf': 200, 'l2_regularization': 0.04605, 'max_features': 0.520807, 'max_iter': 159}
Δ vs configuración 1.0.0 (misma CV): {'mean': -0.00365, 'se': 0.00263, 'per_fold': [-0.01658, -0.00116, 0.00107, -0.00108, -0.00241, -0.00178], 'better_by_1se': True}


,count,min,50%,max
family,,,,
hgb,60.0,0.2924,0.2971,0.3387
hgb_v1,1.0,0.2964,0.2964,0.2964
random_forest,4.0,0.3018,0.3058,0.3084
spline_logistic,2.0,0.3120,0.3123,0.3125


## 4. Qué aprendió el modelo

Importancia por permutación (aumento del log loss al barajar cada feature en los folds de validación, IC 95 % entre
folds × repeticiones) y dependencia parcial de las cuatro principales, contrastada con la física de la lluvia.

![Permutación](../reports/figures/study_permutation_importance.png)
![PDP](../reports/figures/study_partial_dependence.png)

In [4]:
imp = pd.DataFrame(study["permutation_importance"]).set_index("feature")
pdp = {f: d["trend"] for f, d in study["partial_dependence"].items()}
display(imp.head(10))
pdp

,mean,ci95
feature,,
hour_cos,0.0699,"[0.0308, 0.1085]"
hour_sin,0.0508,"[0.0313, 0.068]"
precipitation_5h_log1p,0.0185,"[0.0109, 0.0331]"
dew_point_depression,0.0073,"[0.0009, 0.0161]"
wind_u,0.0070,"[0.0023, 0.0145]"
longitude,0.0065,"[0.0038, 0.0089]"
cape_prev1_log1p,0.0061,"[0.0025, 0.0123]"
latitude,0.0047,"[0.0025, 0.0095]"
precipitation_prev2_log1p,0.0027,"[-0.0014, 0.0052]"


{'precipitation_5h_log1p': 'creciente',
 'dew_point_depression': 'decreciente',
 'wind_u': 'creciente',
 'longitude': 'no monotónica'}

## 5. Ablación y curva de aprendizaje

![Ablación](../reports/figures/study_ablation.png)
![Curva de aprendizaje](../reports/figures/study_learning_curve.png)

In [5]:
ab = pd.DataFrame({g: {"Δ log loss": v["delta_vs_full"]["mean"], "EE": v["delta_vs_full"]["se"]}
                   for g, v in study["ablation"].items()}).T.sort_values("Δ log loss", ascending=False)
display(ab)
pd.DataFrame(study["learning_curve"])[["train_months", "mean_log_loss"]]

,Δ log loss,EE
hora del día,0.0143,0.0014
ubicación,0.0075,0.0010
ventana del pronóstico,0.0062,0.0011
termodinámica,0.0047,0.0024
viento,0.0044,0.0010
estacionalidad,0.0023,0.0030
precipitación pronosticada,0.0016,0.0009
nubosidad,0.0013,0.0002
inestabilidad (CAPE),0.0010,0.0004
presión,0.0006,0.0004


,train_months,mean_log_loss
0,3,0.5110
1,6,0.4214
2,12,0.3467
3,18,0.3413
4,todo,0.3373


## 6. Dónde falla (predicciones *out-of-fold*)

In [6]:
for dim in ("season", "hour_local", "forecast_precip_mm"):
    display(pd.DataFrame(study["errors_oof"][dim]).set_index("segment"))

,n,rain_rate,mean_pred,roc_auc,brier,ece
segment,,,,,,
lluviosa,75816,0.2045,0.2360,0.8491,0.1174,0.0316
seca,86112,0.1133,0.1244,0.8877,0.0699,0.0121
transición,9672,0.0641,0.1318,0.8980,0.0516,0.0685


,n,rain_rate,mean_pred,roc_auc,brier,ece
segment,,,,,,
0,7150,0.0604,0.0890,0.8109,0.0493,0.0295
1,7150,0.0641,0.0875,0.7965,0.0537,0.0235
2,7150,0.0866,0.1060,0.8405,0.0637,0.0202
3,7150,0.0796,0.0981,0.8518,0.0586,0.0189
4,7150,0.0727,0.0939,0.8740,0.0527,0.0219
5,7150,0.0650,0.0879,0.8722,0.0493,0.0231
6,7150,0.0572,0.0875,0.8637,0.0440,0.0306
7,7150,0.0638,0.0844,0.8624,0.0471,0.0236
8,7150,0.0573,0.0787,0.8934,0.0430,0.0256


,n,rain_rate,mean_pred,roc_auc,brier,ece
segment,,,,,,
"(0, 0.1]",22441,0.2117,0.2380,0.8204,0.1288,0.0263
"(0.1, 0.5]",20244,0.2943,0.3238,0.8051,0.1591,0.0296
"(0.5, 2]",9744,0.4436,0.4584,0.7429,0.2039,0.0196
0,113532,0.0690,0.0920,0.8411,0.0542,0.0230
>2,5639,0.5354,0.5446,0.7312,0.2063,0.0228


## 7. Calibración y umbrales de decisión

Cada fold se calibra solo con las predicciones de los folds anteriores (*cross-fitting* temporal), así que la
comparación no mira el fold evaluado. Con probabilidades calibradas, el umbral de costo mínimo es 1/(1+r) cuando una
lluvia no avisada cuesta r veces una falsa alarma: que el óptimo empírico coincida con el teórico es otra prueba de
calibración.

![Calibración](../reports/figures/study_calibration.png)
![Costo](../reports/figures/study_decision_cost.png)

In [7]:
display(pd.DataFrame({m: {"log loss": d["log_loss"], "ECE": d["ece"]}
                      for m, d in study["calibration"]["methods"].items()}).T)
pd.DataFrame({r: {"umbral teórico": d["theoretical_threshold"], "umbral óptimo": d["optimal_threshold"],
                  "regret 0.40": d["business_thresholds"]["0.4"]["regret_vs_optimal"],
                  "regret 0.70": d["business_thresholds"]["0.7"]["regret_vs_optimal"]}
              for r, d in study["decision"]["ratios"].items()}).T

,log loss,ECE
none,0.3021,0.0273
isotonic,0.3030,0.0052
sigmoid,0.3014,0.0036


,umbral teórico,umbral óptimo,regret 0.40,regret 0.70
1,0.500,0.48,0.0033,0.0135
2,0.333,0.33,0.0032,0.0730
3,0.250,0.27,0.0243,0.1538
4,0.200,0.23,0.0564,0.2455
5,0.167,0.17,0.0976,0.3463


## 8. Evaluación final (una sola vez): prueba 2025 y holdout 2026

IC 95 % por bootstrap en bloques de días. "Supera" = el IC de la diferencia pareada de Brier **y** de log loss
queda por debajo de 0.

![Confiabilidad](../reports/figures/rain_classifier_reliability.png)

In [8]:
for nombre, s in metrics["splits"].items():
    tabla = pd.DataFrame({k: {m: v[m] for m in ("roc_auc", "pr_auc", "brier", "log_loss", "bss", "ece")}
                          for k, v in s["models"].items()}).T
    print(nombre, s["period"][0][:10], "→", s["period"][1][:10], f"· {s['rows']:,} filas", s["claims"])
    display(tabla)

test_2025 2025-01-02 → 2025-12-31 · 113,568 filas {'beats_climatology': True, 'beats_raw_forecast': True, 'beats_logistic': True, 'beats_persistence_reference': False, 'beats_model_1_0_0': True, 'ece_below_0_03': False}


,roc_auc,pr_auc,brier,log_loss,bss,ece
model,0.8456,0.6149,0.1234,0.3934,0.1133,0.0572
climatology,0.7770,0.5052,0.1391,0.4392,0.0000,0.0369
raw_forecast,0.7022,0.3977,0.1567,0.4890,-0.1262,0.0422
logistic,0.8055,0.5452,0.1323,0.4170,0.0490,0.0312
persistence_reference,0.8152,0.5662,0.1247,1.7233,0.1033,0.1248
model_1_0_0,0.8250,0.5805,0.1264,0.4009,0.0913,0.0346


holdout_2026 2026-01-02 → 2026-10-15 · 89,518 filas {'beats_climatology': True, 'beats_raw_forecast': True, 'beats_logistic': True, 'beats_persistence_reference': False, 'beats_model_1_0_0': True, 'ece_below_0_03': False}


,roc_auc,pr_auc,brier,log_loss,bss,ece
model,0.8372,0.5247,0.1049,0.3422,0.1507,0.0389
climatology,0.7236,0.3620,0.1235,0.4030,0.0000,0.0382
raw_forecast,0.7141,0.3457,0.1251,0.4109,-0.0135,0.0279
logistic,0.7988,0.4694,0.1098,0.3593,0.1110,0.0169
persistence_reference,0.8064,0.5097,0.1054,1.4568,0.1461,0.1055
model_1_0_0,0.8054,0.4877,0.1087,0.3565,0.1200,0.0206


## 9. Registro de decisiones

Qué se decidió, con qué evidencia y con qué datos. Ninguna fila usa el holdout 2026.

In [9]:
pd.DataFrame(metrics["decision_log"])

,decision,evidence,data
0,ventana del pronóstico adoptada,study.json:feature_candidates.marginal,CV temporal desarrollo 2022–2024
1,ubicación (lat/lon) adoptada,"study.json:feature_candidates.marginal, spatia...",CV temporal y espacial (dejar una celda ERA5 f...
2,"hgb {'learning_rate': 0.097271, 'max_leaf_node...",study.json:search,CV desarrollo 2022–2024
3,calibración = sigmoid,study.json:calibration,cross-fitting temporal sobre OOF de desarrollo
4,umbrales 0.40/0.70 (RF-10) contrastados con el...,study.json:decision,OOF calibrado de desarrollo (folds 2–6)
5,modelo final hgb entrenado en todo desarrollo ...,metadata.json,desarrollo 2022–2024
6,evaluación única en 2025 y en el holdout fresc...,metrics.json:splits,2025 (ya visto por 1.0.0) y 2026-01 → 2026-10-15


## Limitaciones

La etiqueta es reanálisis ERA5 (~25 km), no lluvia medida: frente a pluviómetros SIATA la concordancia horaria es baja
con cualquier fuente de malla (`reports/metrics/label_validation.json`). Las 13 estaciones caen en 5 celdas: hay muchas
menos observaciones independientes que filas, por eso los IC remuestrean días completos. Detalle en la model card
(`models/rain_classifier/<versión>/model_card.md`).